[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

# Model Metadata and Versioning — Deep Dive

Metadata turns an anonymous blob of bytes into an **auditable artifact**: who built it, for which ONNX IR level, against which operator sets, and with what semantic version. This notebook provides a rigorous treatment of ONNX's multi-axis versioning system, metadata best practices, and CI/CD integration patterns.

| # | Section | Description |
|---|---------|-------------|
| 1 | [Metadata Fields Overview](#1-metadata-fields-overview) | All metadata fields and their purposes |
| 2 | [The Three Versioning Axes](#2-the-three-versioning-axes) | IR version, opset version, model version |
| 3 | [IR Version — Schema Evolution](#3-ir-version--schema-evolution) | What changes between IR versions |
| 4 | [OpSet Versioning — Operator Contracts](#4-opset-versioning--operator-contracts) | Domain-scoped operator versioning |
| 5 | [OpSet Compatibility Matrix](#5-opset-compatibility-matrix) | Which runtimes support which opsets |
| 6 | [Semantic Versioning for Models](#6-semantic-versioning-for-models) | model_version and domain conventions |
| 7 | [metadata_props — Key-Value Storage](#7-metadata_props--key-value-storage) | Extensible metadata |
| 8 | [Version Audit and Validation](#8-version-audit-and-validation) | Automated checking patterns |
| 9 | [CI/CD Integration](#9-cicd-integration) | Versioning in production pipelines |
| 10 | [Key Takeaways](#10-key-takeaways) | Summary |

In [ ]:
# !pip install onnx numpy matplotlib --quiet

import onnx
from onnx import helper, TensorProto, checker, defs, numpy_helper
import numpy as np
import matplotlib.pyplot as plt

## 1. Metadata Fields Overview

ONNX models carry metadata at multiple levels of the Protobuf hierarchy. The primary metadata lives in `ModelProto`:

### ModelProto Metadata Fields

| Field | Proto Tag | Type | Purpose | Example |
|-------|-----------|------|---------|--------|
| `ir_version` | 1 | `int64` | ONNX IR spec version | `9` |
| `producer_name` | 2 | `string` | Exporting tool name | `"pytorch"`, `"tf2onnx"` |
| `producer_version` | 3 | `string` | Exporting tool version | `"2.1.0"` |
| `domain` | 4 | `string` | Model namespace (reverse-DNS) | `"com.example.vision"` |
| `model_version` | 5 | `int64` | User-defined version | `3` |
| `doc_string` | 6 | `string` | Human-readable documentation | `"ResNet-50 for ImageNet"` |
| `opset_import` | 8 | `OperatorSetIdProto[]` | Operator set declarations | `[("", 17)]` |
| `metadata_props` | 14 | `StringStringEntryProto[]` | Extensible key-value pairs | `{"author": "Ada"}` |

### Metadata at Other Levels

```
ModelProto
├── doc_string             ← model-level documentation
├── metadata_props[]       ← model-level key-value pairs
│
├── graph: GraphProto
│   ├── doc_string         ← graph-level documentation
│   │
│   ├── node[]: NodeProto
│   │   └── doc_string     ← per-node documentation
│   │
│   ├── input[]: ValueInfoProto
│   │   └── doc_string     ← per-input documentation
│   │
│   └── output[]: ValueInfoProto
│       └── doc_string     ← per-output documentation
│
└── functions[]: FunctionProto
    └── doc_string         ← per-function documentation
```

The `doc_string` field is available on nearly every Protobuf message type, enabling comprehensive documentation at every level of the model hierarchy.

In [ ]:
a = helper.make_tensor_value_info("A", TensorProto.FLOAT, ["batch", 4])
b = helper.make_tensor_value_info("B", TensorProto.FLOAT, ["batch", 4])
c = helper.make_tensor_value_info("C", TensorProto.FLOAT, ["batch", 4])
node = helper.make_node("Add", ["A", "B"], ["C"])
graph = helper.make_graph([node], "demo", inputs=[a, b], outputs=[c])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])

model.doc_string = "ResNet-50 exported for INT8 edge benchmark."
model.domain = "com.example.vision"
model.model_version = 3
model.producer_name = "custom-exporter"
model.producer_version = "2.4.1"

helper.set_model_props(model, {
    "author": "Ada Lovelace",
    "license": "Apache-2.0",
    "commit": "a1b2c3d",
    "dataset": "ImageNet-1K",
    "accuracy_top1": "76.13",
    "quantization": "INT8 symmetric",
})

checker.check_model(model)

print("=== Model Metadata ===")
print(f"  ir_version:       {model.ir_version}")
print(f"  producer_name:    {model.producer_name}")
print(f"  producer_version: {model.producer_version}")
print(f"  domain:           {model.domain}")
print(f"  model_version:    {model.model_version}")
print(f"  doc_string:       {model.doc_string}")
print(f"  opset_import:")
for oi in model.opset_import:
    dom = oi.domain if oi.domain else '"" (default ai.onnx)'
    print(f"    domain={dom}, version={oi.version}")
print(f"  metadata_props ({len(model.metadata_props)}):")
for p in model.metadata_props:
    print(f"    {p.key} = {p.value}")

## 2. The Three Versioning Axes

ONNX employs a **three-axis versioning system** where each axis is independent and serves a different purpose:

### Versioning Architecture

```
                        ┌─────────────────────────────────────────┐
                        │           ONNX Model Versioning         │
                        └───────────────┬─────────────────────────┘
                                        │
                  ┌─────────────────────┼──────────────────────┐
                  │                     │                      │
           ┌──────┴──────┐     ┌────────┴────────┐    ┌───────┴───────┐
           │ IR Version  │     │ OpSet Version   │    │ Model Version │
           │ (axis 1)    │     │ (axis 2)        │    │ (axis 3)      │
           ├─────────────┤     ├─────────────────┤    ├───────────────┤
           │ Proto schema│     │ Per-domain       │    │ User-defined  │
           │ structure   │     │ operator defs    │    │ iteration     │
           │ changes     │     │ changes          │    │ tracking      │
           │             │     │                  │    │               │
           │ Bumped      │     │ Bumped when ops  │    │ Set by user   │
           │ rarely      │     │ added/modified   │    │ as needed     │
           └─────────────┘     └─────────────────┘    └───────────────┘
```

### Formal Compatibility Predicate

A runtime $R$ can execute model $M$ iff:

$$\text{Compatible}(R, M) = \underbrace{\text{IR}_{R} \geq \text{IR}_{M}}_{\text{schema compat}} \;\wedge\; \underbrace{\bigwedge_{(d,v) \in M.\text{opset}} \text{supports}(R, d, v)}_{\text{operator compat}}$$

The model version axis ($M.\text{model\_version}$) does not affect compatibility — it is purely informational.

### Key Distinction

| Axis | Scope | Who bumps it | Breaking? |
|------|-------|-------------|----------|
| IR version | Proto schema (message structure) | ONNX committee | Potentially |
| OpSet version | Operator semantics (per domain) | ONNX committee | Usually backward-compatible |
| Model version | User's model artifact | Model author | Never (informational) |

## 3. IR Version — Schema Evolution

The **IR (Intermediate Representation) version** tracks changes to the Protobuf schema itself — the set of messages, fields, and their semantics.

### IR Version History

| IR Version | Notable Changes |
|-----------|----------------|
| 1–3 | Early development, rapid iteration |
| 4 | Added `metadata_props`, stabilized `TypeProto` |
| 5 | Added `TrainingInfoProto` |
| 6 | Added `SparseTensorProto` |
| 7 | Added `FunctionProto` for model-local functions |
| 8 | Added `TypeProto.optional`, sequence/map types |
| 9 | Added `TypeProto.SparseTensorType` |
| 10 | Latest stable version |

### Compatibility Rule

A parser compiled for IR version $v_{\text{parser}}$ can parse a model with IR version $v_{\text{model}}$ iff:

$$v_{\text{parser}} \geq v_{\text{model}}$$

Forward compatibility (old parser, new model) works because Protobuf preserves unknown fields. However, the parser won't *understand* new fields — it will serialize them back correctly but cannot interpret their semantics.

### Semantic Versioning Constraints

ONNX IR versions follow a monotonically increasing sequence. Each version $v$ is a superset of version $v-1$:

$$\text{Schema}(v) \supseteq \text{Schema}(v-1) \quad \forall v > 1$$

Fields are never removed, only added. Field numbers are never reused. This ensures backward compatibility.

In [ ]:
print(f"Current ONNX IR version: {onnx.IR_VERSION}")
print(f"Installed default opset: {defs.onnx_opset_version()}")

print("\nIR Version vs OpSet — independent axes:")
ir_opset_pairs = []
for opset in [13, 15, 17, defs.onnx_opset_version()]:
    try:
        m = helper.make_model(
            helper.make_graph(
                [helper.make_node("Relu", ["X"], ["Y"])],
                "test",
                [helper.make_tensor_value_info("X", TensorProto.FLOAT, [1])],
                [helper.make_tensor_value_info("Y", TensorProto.FLOAT, [1])],
            ),
            opset_imports=[helper.make_opsetid("", opset)],
        )
        checker.check_model(m)
        ir_opset_pairs.append((m.ir_version, opset))
        print(f"  opset={opset:3d} -> ir_version={m.ir_version}  [VALID]")
    except Exception as e:
        print(f"  opset={opset:3d} -> ERROR: {e}")

## 4. OpSet Versioning — Operator Contracts

Each `opset_import` entry declares a **(domain, version)** pair that selects a frozen set of operator schemas:

$$\text{OpSetImport} = \{(d_1, v_1), (d_2, v_2), \ldots, (d_k, v_k)\}$$

### Domain System

| Domain string | Canonical name | Content |
|--------------|---------------|--------|
| `""` (empty) | `ai.onnx` | Standard neural network operators |
| `"ai.onnx.ml"` | `ai.onnx.ml` | Classical ML operators (trees, scalers) |
| `"ai.onnx.preview.training"` | Training preview | Gradient, optimizer ops |
| `"com.example.*"` | Custom | User-defined operator domains |

### Version Resolution Algorithm

When a runtime encounters a `NodeProto` with `(domain=d, op_type=t)`, the resolution is:

```
1. Find the imported version: v_d = opset_import[d].version
2. Look up the schema: schema = get_schema(t, v_d, d)
3. The schema used is the one with the highest since_version <= v_d
```

Formally, the effective schema version for operator $t$ in domain $d$ is:

$$v_{\text{eff}}(t, d) = \max\{v_s \mid v_s \leq v_d \wedge \text{schema\_exists}(t, v_s, d)\}$$

### Domain Precedence

If a node has `domain=""`, it uses the default ONNX domain. If `domain` is a non-empty string, it must match one of the imported domains. There is no fallback — missing domain imports are invalid.

$$\forall n \in G.\text{nodes}: \quad n.\text{domain} \in \{d \mid (d, \_) \in M.\text{opset\_import}\}$$

In [ ]:
print("=== OpSet Version Resolution ===")
print(f"Installed default opset: {defs.onnx_opset_version()}\n")

ops_to_check = ["Relu", "Conv", "MatMul", "Softmax", "BatchNormalization",
                "LayerNormalization", "GroupNormalization", "Gelu"]

print(f"{'Operator':<25s} {'since_version':>14s} {'Current schema':>15s}")
print("-" * 58)
for op in ops_to_check:
    try:
        schema = defs.get_schema(op, defs.onnx_opset_version(), "")
        print(f"{op:<25s} {schema.since_version:>14d} {'available':>15s}")
    except Exception:
        print(f"{op:<25s} {'N/A':>14s} {'not in this opset':>15s}")

print("\n=== Softmax schema evolution (semantics changed at opset 13) ===")
for v in [1, 11, 13, 17]:
    try:
        s = defs.get_schema("Softmax", v, "")
        attrs = list(s.attributes.keys())
        print(f"  opset {v:2d}: since_version={s.since_version}, attrs={attrs}")
    except Exception:
        print(f"  opset {v:2d}: not available")

## 5. OpSet Compatibility Matrix

Understanding which opset versions are supported by which runtimes is critical for deployment.

### ONNX Runtime Support Matrix (representative)

```
ONNX Runtime    │  Min Opset  │  Max Opset  │  IR Version
────────────────┼─────────────┼─────────────┼────────────
  1.12.x        │     7       │     17      │    8
  1.14.x        │     7       │     18      │    9
  1.16.x        │     7       │     19      │    9
  1.17.x        │     7       │     20      │    9
  1.18.x        │     7       │     21      │   10
```

### Compatibility Decision Tree

```
Model requires opset V_model
       │
       ▼
  V_model <= V_runtime_max?
       │
  ┌────┴─────┐
  │ YES      │ NO
  ▼          ▼
  Check IR   Upgrade runtime
  version    OR downgrade model
  │          via version_converter
  ▼
  All ops in model
  exist at V_model?
  │
  ┌────┴─────┐
  │ YES      │ NO
  ▼          ▼
  COMPATIBLE  Check for custom ops
              or missing domain import
```

### Best Practice: Choose the Lowest Compatible OpSet

$$v_{\text{target}} = \min\{v \mid \forall n \in G.\text{nodes}: \text{since\_version}(n.\text{op\_type}) \leq v\}$$

This maximizes runtime compatibility while ensuring all operators are available.

In [ ]:
current_opset = defs.onnx_opset_version()

common_ops = ["Relu", "Conv", "MatMul", "Softmax", "Add", "Reshape",
              "Transpose", "Concat", "Gather", "Unsqueeze"]

print("OpSet availability for common operators:")
print(f"{'Op':<20s}", end="")
opset_range = range(7, min(current_opset + 1, 22))
for v in opset_range:
    print(f" {v:>3d}", end="")
print()
print("-" * (20 + 4 * len(list(opset_range))))

for op in common_ops:
    print(f"{op:<20s}", end="")
    for v in opset_range:
        try:
            defs.get_schema(op, v, "")
            print("  OK", end="")
        except Exception:
            print("   -", end="")
    print()

## 6. Semantic Versioning for Models

While ONNX doesn't enforce semantic versioning (SemVer) for `model_version`, adopting SemVer conventions helps with model registry management.

### Recommended Convention

Since `model_version` is a single `int64`, encode major.minor.patch as:

$$\text{model\_version} = \text{major} \times 10000 + \text{minor} \times 100 + \text{patch}$$

For example, version 2.4.1 becomes:

$$\text{model\_version} = 2 \times 10000 + 4 \times 100 + 1 = 20401$$

### Domain + Version as Identity

The pair `(domain, model_version)` uniquely identifies a model variant:

$$\text{ModelID} = (\text{domain}, \text{model\_version}) = (\text{"com.example.vision.resnet50"}, 20401)$$

### SemVer Constraints for ONNX Models

| Change type | SemVer bump | Example |
|------------|------------|--------|
| Input/output shape change | **Major** | Changed input from [1,3,224,224] to [1,3,256,256] |
| Input/output name change | **Major** | Renamed "image" to "pixel_values" |
| Added output | **Minor** | Added auxiliary loss output |
| Weight update (retrained) | **Minor** | Finetuned on new dataset |
| Metadata-only change | **Patch** | Updated doc_string |
| Quantization variant | **Minor** | Created INT8 version of float model |

In [ ]:
def encode_semver(major, minor, patch):
    """Encode semantic version as int64."""
    return major * 10000 + minor * 100 + patch

def decode_semver(version_int):
    """Decode int64 to semantic version tuple."""
    major = version_int // 10000
    minor = (version_int % 10000) // 100
    patch = version_int % 100
    return major, minor, patch

def format_semver(version_int):
    major, minor, patch = decode_semver(version_int)
    return f"{major}.{minor}.{patch}"

test_versions = [
    (1, 0, 0), (1, 2, 3), (2, 4, 1), (3, 0, 0), (10, 5, 99)
]

print(f"{'SemVer':<15s} {'Encoded':>10s} {'Decoded':<15s} {'Match':>6s}")
print("-" * 50)
for major, minor, patch in test_versions:
    encoded = encode_semver(major, minor, patch)
    decoded = format_semver(encoded)
    original = f"{major}.{minor}.{patch}"
    match = "OK" if decoded == original else "FAIL"
    print(f"{original:<15s} {encoded:>10d} {decoded:<15s} {match:>6s}")

model.domain = "com.example.vision.resnet50"
model.model_version = encode_semver(2, 4, 1)
print(f"\nModel identity: ({model.domain}, v{format_semver(model.model_version)})")

## 7. metadata_props — Key-Value Storage

`metadata_props` is an extensible key-value store using `StringStringEntryProto` messages. Since both keys and values are strings, numeric values must be string-encoded.

### Recommended Keys

| Key | Purpose | Example value |
|-----|---------|---------------|
| `author` | Model creator | `"Ada Lovelace"` |
| `license` | License identifier | `"Apache-2.0"` |
| `commit` | Source control hash | `"a1b2c3d4e5f6"` |
| `dataset` | Training data | `"ImageNet-1K"` |
| `accuracy_top1` | Metric value | `"76.13"` |
| `accuracy_top5` | Metric value | `"92.86"` |
| `quantization` | Quant method | `"INT8 symmetric"` |
| `input_normalization` | Preprocessing | `"mean=[0.485,0.456,0.406] std=[0.229,0.224,0.225]"` |
| `training_epochs` | Training config | `"90"` |
| `learning_rate` | Training config | `"0.1"` |
| `created_at` | Timestamp | `"2024-01-15T10:30:00Z"` |
| `export_tool` | Converter details | `"torch.onnx.export+opset17"` |

### Programmatic Access

```python
# Write
helper.set_model_props(model, {"key": "value", ...})

# Read
props = {p.key: p.value for p in model.metadata_props}

# Update (append)
entry = model.metadata_props.add()
entry.key = "new_key"
entry.value = "new_value"
```

In [ ]:
from datetime import datetime

comprehensive_metadata = {
    "author": "ML Engineering Team",
    "license": "MIT",
    "commit": "abc123def456",
    "dataset": "CIFAR-10",
    "accuracy_test": "95.2",
    "training_epochs": "200",
    "optimizer": "AdamW",
    "learning_rate": "3e-4",
    "weight_decay": "0.01",
    "batch_size": "256",
    "created_at": datetime.now().isoformat(),
    "framework": "PyTorch 2.1",
    "export_tool": "torch.onnx.export",
    "target_hardware": "NVIDIA T4 GPU",
    "input_format": "NCHW float32 [0,1] normalized",
}

del model.metadata_props[:]
helper.set_model_props(model, comprehensive_metadata)

print(f"Total metadata entries: {len(model.metadata_props)}\n")
props = {p.key: p.value for p in model.metadata_props}
max_key_len = max(len(k) for k in props)
for key, value in sorted(props.items()):
    print(f"  {key:<{max_key_len}} = {value}")

print(f"\nSerialize overhead from metadata: "
      f"{sum(len(p.key) + len(p.value) + 6 for p in model.metadata_props)} bytes (approx)")

## 8. Version Audit and Validation

A **version audit** checks that a model's declared versions are internally consistent and compatible with the target deployment environment.

### Audit Checks

1. **IR version support**: Is the model's IR version supported by the runtime?
2. **OpSet availability**: Are all imported opsets supported?
3. **Operator coverage**: Do all node op_types exist in the imported opsets?
4. **Producer consistency**: Does the producer name/version match expectations?
5. **Metadata completeness**: Are required metadata keys present?

### Formal Validation Predicate

$$\text{Valid}(M) = \text{check\_ir}(M) \wedge \text{check\_opsets}(M) \wedge \text{check\_ops}(M) \wedge \text{check\_graph}(M)$$

In [ ]:
def full_version_audit(m: onnx.ModelProto, required_keys=None) -> dict:
    """Comprehensive version and metadata audit."""
    report = {"passed": True, "checks": []}

    report["checks"].append({
        "name": "IR version",
        "value": m.ir_version,
        "max_supported": onnx.IR_VERSION,
        "ok": m.ir_version <= onnx.IR_VERSION,
    })

    for oi in m.opset_import:
        domain = oi.domain or "ai.onnx (default)"
        report["checks"].append({
            "name": f"OpSet [{domain}]",
            "value": oi.version,
            "ok": True,
        })

    imported_domains = {oi.domain for oi in m.opset_import}
    for node in m.graph.node:
        if node.domain not in imported_domains:
            report["checks"].append({
                "name": f"Missing domain for {node.op_type}",
                "value": node.domain,
                "ok": False,
            })
            report["passed"] = False

    report["checks"].append({
        "name": "Producer",
        "value": f"{m.producer_name} v{m.producer_version}",
        "ok": bool(m.producer_name),
    })

    report["checks"].append({
        "name": "Model identity",
        "value": f"domain={m.domain!r} version={m.model_version}",
        "ok": bool(m.domain) and m.model_version > 0,
    })

    if required_keys:
        props = {p.key for p in m.metadata_props}
        missing = set(required_keys) - props
        report["checks"].append({
            "name": "Required metadata",
            "value": f"present={len(props)}, missing={missing or 'none'}",
            "ok": len(missing) == 0,
        })
        if missing:
            report["passed"] = False

    try:
        checker.check_model(m)
        report["checks"].append({"name": "onnx.checker", "value": "PASSED", "ok": True})
    except Exception as e:
        report["checks"].append({"name": "onnx.checker", "value": str(e)[:80], "ok": False})
        report["passed"] = False

    return report

required = ["author", "license", "commit", "dataset"]
report = full_version_audit(model, required_keys=required)

print("=== Version Audit Report ===")
for check in report["checks"]:
    status = "PASS" if check["ok"] else "FAIL"
    print(f"  [{status}] {check['name']}: {check['value']}")
print(f"\nOverall: {'PASSED' if report['passed'] else 'FAILED'}")

## 9. CI/CD Integration

### Automated Versioning Pipeline

```
┌───────────┐    ┌──────────────┐    ┌─────────────┐    ┌──────────────┐
│  Training  │───▶│  Export to   │───▶│  Version    │───▶│  Registry    │
│  Script    │    │  ONNX        │    │  Audit      │    │  Upload      │
│            │    │  (set meta)  │    │  (validate) │    │  (if pass)   │
└───────────┘    └──────────────┘    └─────────────┘    └──────────────┘
                       │                     │                  │
                       ▼                     ▼                  ▼
                  Set producer,         Check opset,       Push to model
                  commit hash,         IR version,        store with
                  metrics              required keys      version tag
```

### Best Practices for Production

1. **Pin exporter versions** in CI and record in `producer_version`
2. **Choose target opset deliberately** — use the lowest opset your deployment supports
3. **Set domain + model_version** consistently with your model registry
4. **Use metadata_props** for commit hash, dataset fingerprint, quantization recipe
5. **Keep doc_string** updated with deployment notes (input resolutions, normalization)
6. **After bumping opset**, rerun checker and accuracy regression tests
7. **Gate deployments** on version audit passing

In [ ]:
def ci_export_model(model, version_tuple, commit_hash, metrics):
    """Simulate a CI/CD model export with full metadata."""
    major, minor, patch = version_tuple
    model.model_version = encode_semver(major, minor, patch)
    model.producer_name = "ci-pipeline"
    model.producer_version = "3.0.0"
    model.domain = "com.example.production.classifier"

    del model.metadata_props[:]
    meta = {
        "commit": commit_hash,
        "ci_job_id": "build-12345",
        "created_at": datetime.now().isoformat(),
    }
    meta.update({f"metric_{k}": str(v) for k, v in metrics.items()})
    helper.set_model_props(model, meta)

    checker.check_model(model)
    return model

exported = ci_export_model(
    model,
    version_tuple=(3, 1, 0),
    commit_hash="deadbeef12345678",
    metrics={"accuracy": 0.952, "f1_score": 0.948, "latency_ms": 12.3},
)

print("=== CI Export Result ===")
print(f"  Model: {exported.domain} v{format_semver(exported.model_version)}")
print(f"  Producer: {exported.producer_name} v{exported.producer_version}")
print(f"  Metadata:")
for p in exported.metadata_props:
    print(f"    {p.key} = {p.value}")

required_ci_keys = ["commit", "ci_job_id", "created_at"]
report = full_version_audit(exported, required_keys=required_ci_keys)
print(f"\n  Audit: {'PASSED' if report['passed'] else 'FAILED'}")

## 10. Key Takeaways

1. **ONNX uses three independent versioning axes**: IR version (proto schema), OpSet version (operator semantics per domain), and model version (user-defined iteration tracking).

2. **Compatibility is determined by**: $\text{Compatible}(R, M) = (\text{IR}_R \geq \text{IR}_M) \wedge \bigwedge_{(d,v)} \text{supports}(R, d, v)$

3. **IR versions are monotonically increasing** and backward-compatible: $\text{Schema}(v) \supseteq \text{Schema}(v-1)$. They change rarely and control the proto message structure.

4. **OpSet versions** are per-domain. The effective schema for an operator is the highest `since_version` $\leq$ the imported version: $v_{\text{eff}} = \max\{v_s \leq v_d\}$.

5. **Semantic versioning** can be encoded in `model_version` as: $\text{major} \times 10000 + \text{minor} \times 100 + \text{patch}$.

6. **`metadata_props`** provides extensible key-value storage for provenance (commit hash, author), metrics (accuracy, latency), and deployment info (quantization, hardware target).

7. **Version audits** should be automated in CI/CD: check IR compatibility, opset coverage, required metadata keys, and run `onnx.checker.check_model()`.

8. **Best practice**: choose the lowest opset version that supports all operators in your model, maximizing deployment compatibility across runtime versions.